In [2]:
from pathlib import Path
import pandas as pd
import osmnx as ox

In [ ]:
# Disable OSMnx caching for this geocoding run
ox.settings.use_cache = False

input_csv = Path("clients.csv")
output_csv = Path("client_coords.csv")

df = pd.read_csv(input_csv).copy()

def geocode_with_fallback(address: str):
    if pd.isna(address) or str(address).strip() == "":
        return pd.Series({"latitude": 0.0, "longitude": 0.0, "geocode_ok": False})

    query = str(address)
    if "netherlands" not in query.lower():
        query = f"{query}, Heerlen, Netherlands"

    try:
        lat, lon = ox.geocode(query)
        return pd.Series({"latitude": float(lat), "longitude": float(lon), "geocode_ok": True})
    except Exception:
        return pd.Series({"latitude": 0.0, "longitude": 0.0, "geocode_ok": False})

coords = df["address"].apply(geocode_with_fallback)
client_coords = pd.concat([df, coords], axis=1)

client_coords.to_csv(output_csv, index=False)

print(f"Saved: {output_csv.resolve()}")
print(f"Total rows: {len(client_coords)}")
print(f"Geocoded OK: {int(client_coords['geocode_ok'].sum())}")
print(f"Fallback 0/0 rows: {int((~client_coords['geocode_ok']).sum())}")
client_coords.head(5)

KeyboardInterrupt: 

In [4]:
# Import the client_coords and display all with lat and long 0
client_coords = pd.read_csv(output_csv)
fallback_rows = client_coords[~client_coords["geocode_ok"]]
print(f"Total fallback rows: {len(fallback_rows)}")

# print all street names with lat long 0
print("Fallback addresses:")
for idx, row in fallback_rows.iterrows():
    print(f" - {row['address']}")
    

Total fallback rows: 11
Fallback addresses:
 - Julianaweg 57 6413 FI Heerlen
 - Lindelaan 14 6414 XW Heerlen
 - Lienaertsstraat 21 6416 YR Heerlen
 - Lindelaan 50 6414 JO Heerlen
 - Lindelaan 18 6414 LH Heerlen
 - Lindelaan 8 6414 MB Heerlen
 - Stationsstraat 17 6411 HU Heerlen
 - Lindelaan 21 6414 QP Heerlen
 - Beukenlaan 4 6414 CS Heerlen
 - Stationsstraat 4 6411 YI Heerlen
 - Julianaweg 32 6413 OD Heerlen
